Install Packages

In [140]:
# Install packages
# %pip install -r ../../requirements.txt

In [141]:
# Install scispacy model for classification (inside .venv)
# pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz

In [142]:
import datasets
import pandas as pd
import numpy as np
import spacy
import scispacy
import torch
from transformers import pipeline

print("datasets:", datasets.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("spacy:", spacy.__version__)
print("scispacy:", scispacy.__version__)
print("torch:", torch.__version__)

datasets: 5.0.1
pandas: 3.0.5
numpy: 1.26.4
spacy: 3.7.5
scispacy: 0.6.2
torch: 2.14.0


In [143]:
from datasets import load_dataset

unlabeled_data = load_dataset("qiaojin/PubMedQA", "pqa_unlabeled", split="train")
labeled_data = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")

In [144]:
print(unlabeled_data)
print(unlabeled_data[0])
print(labeled_data)
print(labeled_data[0])

Dataset({
    features: ['pubid', 'question', 'context', 'long_answer'],
    num_rows: 61249
})
{'pubid': 14499029, 'question': 'Is naturopathy as effective as conventional therapy for treatment of menopausal symptoms?', 'context': {'contexts': ['Although the use of alternative medicine in the United States is increasing, no published studies have documented the effectiveness of naturopathy for treatment of menopausal symptoms compared to women receiving conventional therapy in the clinical setting.', 'To compare naturopathic therapy with conventional medical therapy for treatment of selected menopausal symptoms.', 'A retrospective cohort study, using abstracted data from medical charts.', 'One natural medicine and six conventional medical clinics at Community Health Centers of King County, Washington, from November 1, 1996, through July 31, 1998.', 'Women aged 40 years of age or more with a diagnosis of menopausal symptoms documented by a naturopathic or conventional physician.', 'Imp

In [145]:
# Before using the labeled set as your gold evaluation set, check whether its questions/pubids overlap with the unlabeled pool. If there is overlap, we must exclude those rows from your training pool.
labeled_ids = set(labeled_data["pubid"])
unlabeled_ids = set(unlabeled_data["pubid"])

overlap = labeled_ids.intersection(unlabeled_ids)

print("Overlap:", len(overlap))
gold_eval_set = labeled_data
# Remove any gold-eval examples from the unlabeled training pool
if len(overlap) > 0:
    unlabeled_data = unlabeled_data.filter(
        lambda row: row["pubid"] not in overlap
    )
print(f"Gold eval: {len(gold_eval_set)}")

Overlap: 0
Gold eval: 1000


In [146]:
# Create human training pool
SEED = 42

unlabeled_data_shuffled = unlabeled_data.shuffle(seed=SEED)

G0_BASE_SIZE = 15000
ANCHOR_POOL_SIZE = 15000

g0_human_train = unlabeled_data_shuffled.select(range(G0_BASE_SIZE))

human_anchor_pool = unlabeled_data_shuffled.select(
    range(G0_BASE_SIZE, G0_BASE_SIZE + ANCHOR_POOL_SIZE)
)

print(f"G0 train: {len(g0_human_train)}")
print(f"Anchor pool: {len(human_anchor_pool)}")
print(f"Gold eval: {len(gold_eval_set)}")

G0 train: 15000
Anchor pool: 15000
Gold eval: 1000


In [147]:
print(g0_human_train[0])

{'pubid': 22949290, 'question': 'Unipolar mania: a distinct entity or characteristic of manic preponderance?', 'context': {'contexts': ['It has been reported that fewer patients with unipolar mania respond to lithium prophylaxis as do those with classical bipolar disorder. This study aimed to determine if the difference to response to lithium is related to unipolar mania or to a high preponderance of mania during the course of bipolarity.', 'The study included bipolar-I patients (according to DSM-IV criteria) that had a ≥ 2-year history of either lithium or valproate prophylaxis as monotherapy. The response rate in the patients with unipolar mania and classical bipolar disorder were compared. Then, the response rate to lithium in all the patients with a manic episode rate<50% and>50%, and<80% and>80% during their course were compared. Finally, the above comparisons were repeated, excluding the patients with unipolar mania.', 'The study included 121 bipolar-I patients (34 unipolar mania

In [148]:
# Prompt formatting
def format_pubmed_qa(row):
    ctx = row["context"]
    if isinstance(ctx, dict) and "contexts" in ctx:
        context_str = " ".join(ctx["contexts"])
    elif isinstance(ctx, list):
        context_str = " ".join(ctx)
    else:
        context_str = str(ctx)
    return {
        "text": (
            f"### Question:\n{row['question']}\n\n"
            f"### Context:\n{context_str}\n\n"
            f"### Long Answer:\n{row['long_answer']}"
        )
    }

g0_formatted = g0_human_train.map(format_pubmed_qa)
print("Number of formatted rows:", len(g0_formatted))
print("\n--- SAMPLE ---\n")
print(g0_formatted[0]["text"])

Number of formatted rows: 15000

--- SAMPLE ---

### Question:
Unipolar mania: a distinct entity or characteristic of manic preponderance?

### Context:
It has been reported that fewer patients with unipolar mania respond to lithium prophylaxis as do those with classical bipolar disorder. This study aimed to determine if the difference to response to lithium is related to unipolar mania or to a high preponderance of mania during the course of bipolarity. The study included bipolar-I patients (according to DSM-IV criteria) that had a ≥ 2-year history of either lithium or valproate prophylaxis as monotherapy. The response rate in the patients with unipolar mania and classical bipolar disorder were compared. Then, the response rate to lithium in all the patients with a manic episode rate<50% and>50%, and<80% and>80% during their course were compared. Finally, the above comparisons were repeated, excluding the patients with unipolar mania. The study included 121 bipolar-I patients (34 unip

In [149]:
print(g0_formatted[0])

{'pubid': 22949290, 'question': 'Unipolar mania: a distinct entity or characteristic of manic preponderance?', 'context': {'contexts': ['It has been reported that fewer patients with unipolar mania respond to lithium prophylaxis as do those with classical bipolar disorder. This study aimed to determine if the difference to response to lithium is related to unipolar mania or to a high preponderance of mania during the course of bipolarity.', 'The study included bipolar-I patients (according to DSM-IV criteria) that had a ≥ 2-year history of either lithium or valproate prophylaxis as monotherapy. The response rate in the patients with unipolar mania and classical bipolar disorder were compared. Then, the response rate to lithium in all the patients with a manic episode rate<50% and>50%, and<80% and>80% during their course were compared. Finally, the above comparisons were repeated, excluding the patients with unipolar mania.', 'The study included 121 bipolar-I patients (34 unipolar mania

Load PubMedBERT BC5CDR NER

In [150]:
# Try new classifier
pubmed_ner_pipeline = pipeline(
    "token-classification",
    model="HFpf/pubmedbert-bc5cdr-ner",
    aggregation_strategy="simple"
)

print("PubMedBERT BC5CDR NER loaded!")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5641.24it/s]


PubMedBERT BC5CDR NER loaded!


In [151]:
# B- (Begin) — marks the first token of an entity span.
# I- (Inside) — marks any subsequent token that continues the same entity span.
# O (Outside) — the token isn't part of any entity at all.
LABEL_MAP = {
    "LABEL_0": "O",
    "LABEL_1": "B-Chemical",
    "LABEL_2": "B-Disease",
    "LABEL_3": "I-Disease",
    "LABEL_4": "I-Chemical"
}

def extract_pubmed_entities(text):
    raw_entities = pubmed_ner_pipeline(text)

    entities = []
    current = None

    for entity in raw_entities:
        label = LABEL_MAP[entity["entity_group"]]

        # Outside any entity
        if label == "O":
            if current is not None:
                entities.append(current)
                current = None
            continue

        prefix, entity_type = label.split("-", 1)

        start = entity["start"]
        end = entity["end"]
        score = float(entity["score"])

        # Beginning of a new entity
        if prefix == "B":
            if current is not None:
                entities.append(current)

            current = {
                "entity": text[start:end],
                "type": entity_type,
                "start": start,
                "end": end,
                "scores": [score]
            }

        # Continuation of current entity
        elif prefix == "I":
            if current is not None and current["type"] == entity_type:
                current["entity"] = text[current["start"]:end]
                current["end"] = end
                current["scores"].append(score)

    if current is not None:
        entities.append(current)

    # Average scores across entity pieces
    for entity in entities:
        entity["score"] = (
            sum(entity["scores"]) / len(entity["scores"])
        )
        del entity["scores"]

    return entities

In [152]:
text = (
    "Patients with bipolar disorder were treated "
    "with lithium and valproate."
)
print(g0_formatted[0]["text"])

entities = extract_pubmed_entities(text)

for entity in entities:
    print(
        f"{entity['type']:10s} | "
        f"{entity['entity']:25s} | "
        f"score={entity['score']:.3f}"
    )

### Question:
Unipolar mania: a distinct entity or characteristic of manic preponderance?

### Context:
It has been reported that fewer patients with unipolar mania respond to lithium prophylaxis as do those with classical bipolar disorder. This study aimed to determine if the difference to response to lithium is related to unipolar mania or to a high preponderance of mania during the course of bipolarity. The study included bipolar-I patients (according to DSM-IV criteria) that had a ≥ 2-year history of either lithium or valproate prophylaxis as monotherapy. The response rate in the patients with unipolar mania and classical bipolar disorder were compared. Then, the response rate to lithium in all the patients with a manic episode rate<50% and>50%, and<80% and>80% during their course were compared. Finally, the above comparisons were repeated, excluding the patients with unipolar mania. The study included 121 bipolar-I patients (34 unipolar mania and 87 classical bipolar disorder). Th

In [153]:
for i in range(10):
    text = g0_formatted[i]["long_answer"]
    print(text)
    print(f"\n{'=' * 80}")
    print(f"Example {i}")
    print(f"{'=' * 80}")

    entities = extract_pubmed_entities(text)

    for entity in entities:
        print(
            f"{entity['type']:10s} | "
            f"{entity['entity']:35s} | "
            f"score={entity['score']:.3f}"
        )

Fewer patients with unipolar mania responded to lithium prophylaxis than those with classical bipolar disorder, which appeared to be related to unipolar mania, rather than to a high manic predominance during the disease course. On the other hand, response to valproate prophylaxis was similar in the unipolar mania and classical bipolar disorder groups.

Example 0
Disease    | unipolar mania                      | score=0.985
Chemical   | lithium                             | score=1.000
Disease    | bipolar disorder                    | score=0.999
Disease    | unipolar mania                      | score=0.978
Disease    | manic                               | score=0.661
Chemical   | valproate                           | score=1.000
Disease    | unipolar mania                      | score=0.988
Disease    | bipolar disorder                    | score=0.999
Our findings demonstrate socioeconomic inequalities in AMI incidence in migrant groups and suggest a convergence towards the Dutch 

In [154]:
from transformers import AutoTokenizer
import numpy as np

MODEL_NAME = "Qwen/Qwen2.5-1.5B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

token_lengths = []

for row in g0_formatted:
    token_ids = tokenizer(
        row["text"],
        add_special_tokens=True
    )["input_ids"]

    token_lengths.append(len(token_ids))

In [155]:
print("Min:", np.min(token_lengths))
print("Mean:", np.mean(token_lengths))
print("Median:", np.median(token_lengths))
print("90th percentile:", np.percentile(token_lengths, 90))
print("95th percentile:", np.percentile(token_lengths, 95))
print("99th percentile:", np.percentile(token_lengths, 99))
print("Max:", np.max(token_lengths))

Min: 92
Mean: 396.15313333333336
Median: 386.0
90th percentile: 532.0
95th percentile: 590.0
99th percentile: 733.0100000000002
Max: 1472


In [159]:
for limit in [512, 768, 1024, 1536, 2048]:
    count = sum(length > limit for length in token_lengths)

    print(
        f"> {limit}: "
        f"{count} examples "
        f"({count / len(token_lengths) * 100:.2f}%)"
    )

> 512: 1900 examples (12.67%)
> 768: 108 examples (0.72%)
> 1024: 13 examples (0.09%)
> 1536: 0 examples (0.00%)
> 2048: 0 examples (0.00%)


In [160]:
# Use 768 as 99th percentile is  733
long_indices = [
    i for i, length in enumerate(token_lengths)
    if length > 1024
]

print("Examples over 768:", len(long_indices))

for i in long_indices[:5]:
    print("=" * 80)
    print("Index:", i)
    print("Tokens:", token_lengths[i])
    print()
    print(g0_formatted[i]["text"][:1500])
    print()

Examples over 768: 13
Index: 45
Tokens: 1103

### Question:
Is Pes Cavus Alignment Associated With Lisfranc Injuries of the Foot?

### Context:
Lisfranc (tarsometatarsal joint) injuries are relatively rare, accounting for less than 1% of all fractures, and as many as 20% of subtle Lisfranc injuries are missed at the initial patient presentation. An undiagnosed Lisfranc injury can have devastating consequences to the patient. Therefore, any factor that can raise a clinician's index of suspicion to make this diagnosis is potentially important. The cavus foot has been associated with various maladies of the lower extremity, but to our knowledge, it has not been reported to be associated with Lisfranc injury.QUESTIONS/ Do patients who experience a low-energy Lisfranc injury have greater talar head coverage and a greater talo-first metatarsal angle than control subjects? A retrospective, case-control study was conducted from September 2011 to December 2014 to identify patients diagnosed and

Tokenization

In [162]:
MAX_LENGTH = 1024

def tokenize_function(row):
    return tokenizer(
        row["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False
    )

In [163]:
g0_tokenized = g0_formatted.map(
    tokenize_function,
    batched=False
)

Map: 100%|██████████| 15000/15000 [00:12<00:00, 1231.90 examples/s]


In [164]:
print(g0_tokenized[0].keys())
print("Token count:", len(g0_tokenized[0]["input_ids"]))

dict_keys(['pubid', 'question', 'context', 'long_answer', 'text', 'input_ids', 'attention_mask'])
Token count: 402


In [165]:
lengths_after_tokenization = [
    len(example["input_ids"])
    for example in g0_tokenized
]

print("Min:", min(lengths_after_tokenization))
print("Max:", max(lengths_after_tokenization))
print("Examples:", len(lengths_after_tokenization))

Min: 92
Max: 1024
Examples: 15000


QLoRA Setup